# Control policy optimization

In this example, a symbolic policy is evolved for the pendulum swingup task. Gymnax is used for simulation of the pendulum environment, showing that Kozax can easily be extended to external libraries.

In [ ]:
!pip install brax
!pip install gymnax
!pip install jaxtyping

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.7/232.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 356.9/356.9 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.4/172.4 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.9/20.9 MB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 55.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 741.0/741.0 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.6/86.6 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.0 MB/s eta 0:00:00


In [ ]:
# Specify the cores to use for XLA
import functools
import os
os.environ["XLA_FLAGS"] = '--xla_force_host_platform_device_count=10'
import jax
import jax.numpy as jnp
import jax.random as jr
import jaxlib

#import gymnax
import brax
from brax import envs
import matplotlib.pyplot as plt

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'


In [ ]:
import jax
import jax.numpy as jnp
from brax import envs


class BraxGymnaxWrapper:

    def __init__(self, env_name, backend):
        self.env = envs.get_environment(env_name=env_name, backend=backend)
        self.observation_space = self.env.observation_size
        self.action_space = self.env.action_size

    # state, env_state = self.env.reset(subkey, self.env_params)

    def reset(self, key, params=None):
        #state consists of pipeline_state, obs, reward, done, metrics, info
        state = self.env.reset(key)
        return state.obs, state

     # next_state, next_env_state, reward, done, _ = self.env.step(
     #            subkey, env_state, action, self.env_params
     #        )

    def step(self, state, action, params=None):

      next_state = self.env.step(state, action)

      done = next_state.done

      # Freeze state after done
      obs = jnp.where(done, state.obs, next_state.obs)
      new_state = jax.tree.map(
          lambda new, old: jnp.where(done, old, new),
          next_state,
          state
      )

      reward = jnp.where(done, 0.0, next_state.reward)

      return obs, new_state, reward, done, {}

#Shouldn't states be represented the same in Gymnax and Brax -> env_state only used by respective environments to compute next step. Not explicitly used in Kozax
#EnvState(time=Array(0, dtype=int32, weak_type=True), theta=Array(1.4304566, dtype=float32), theta_dot=Array(0.5757351, dtype=float32), last_u=Array(0., dtype=float32, weak_type=True))) (Gymnax)
#State(pipeline_state, obs, reward, done, metrics, info)
#state.pipeline_state contains: q, qd, x, rot, xd, vel, contact, root_com, oinr, rot, i, mass, cd, vel, cdof, vel, cdofd, vel, mass_mx_inv, con_jac, con_aref, qf_smooth, qdd

#obs is the same for Gymnax and Brax

In [ ]:
from brax.envs.wrappers.training import EpisodeWrapper

In [ ]:
def repeated_evaluation(key, policy, env):
    def single_run(key):
        obs, env_state = env.reset(key)
        (_, _, _), (_, treward, _, _) = jax.lax.scan(
            step_fn(policy),
            (obs, env_state, False),
            jnp.arange(T)
        )
        return jnp.sum(treward)

    keys = jr.split(key, 10)
    return jax.vmap(single_run)(keys)

## Inverted pendulum

Action Space: [-1, 1], where action represents the numerical force applied to the cart

Observation Space: $ℝ^{4}$

0.   car_position
1.   vertical_angle_pole
2.   linear_velocity_cart
3.   angular_velocity_pole

Reward: +1 is awarded for each timestep that the pole is upright.

The episode terminates when episode duration reaches 1000 timestep or the absolute value of the vertical angle between the pole and the cart is greater than 0.2 radians.

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("inverted_pendulum", 'generalized')
T = 1000

/usr/local/lib/python3.12/dist-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        done = done | done_new.astype(bool)
        reward = jnp.where(done, 0.0, reward)

        return (obs, env_state, done), (obs, reward, action, done)

    return _step

def visualize_trajectory(all_obs, treward, actions, dones):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  all_dones = jnp.array(dones)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  reward = jnp.sum(all_rewards)
  all_obs = jnp.array(all_obs)
  plot_list = []

  cart_y = 0.0
  pole_length = 0.5

  for i in range(0, T_term, 50):
    print(
            f"Timestep {i} : "
            f"Cart position: {all_obs[i, 0]} | "
            f"Cart velocity: {all_obs[i, 2]} | "
            f"Pole angle: {all_obs[i, 1]}"
        )
    x = all_obs[i, 0]
    theta = all_obs[i, 1]

    # Pole tip position
    pole_x = x + pole_length * jnp.sin(theta)
    pole_y = cart_y + pole_length * jnp.cos(theta)

    plot_list.append([i, x, pole_x, pole_y])


  print(
          f"Timestep {T_term} : "
          f"Cart position: {all_obs[T_term, 0]} | "
          f"Cart velocity: {all_obs[T_term, 2]} | "
          f"Pole angle: {all_obs[T_term, 1]}"
      )
  x = all_obs[T_term, 0]
  theta = all_obs[T_term, 1]

  # Pole tip position
  pole_x = x + pole_length * jnp.sin(theta)
  pole_y = cart_y + pole_length * jnp.cos(theta)

  plot_list.append([T_term, x, pole_x, pole_y])
  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  fig, axs = plt.subplots(nrows=1, ncols=len(plot_list), figsize=(len(plot_list), 6))
  fig.suptitle("State evolution over time")
  for i in range(len(plot_list)):
    t, x, pole_x, pole_y = plot_list[i]
    axs[i].scatter(x, cart_y, c="black")
    axs[i].plot([x, pole_x], [cart_y, pole_y])
    axs[i].scatter(pole_x, pole_y, c="red")
    axs[i].set_title("t="+ str(t))
    axs[i].set_xlim(-2, 2)
    axs[i].set_ylim(-0.1, 1.0)
    axs[i].set_xlabel("x")
    if i == 0:
      axs[i].set_ylabel("y")
  plt.tight_layout()
  plt.show()

  print("Total reward:", reward)

  plt.plot(all_rewards[:T_term], label="reward")
  plt.plot(all_actions[:T_term], label="actions")
  plt.plot(all_obs[:T_term, 0], label="car position")
  plt.plot(all_obs[:T_term, 1], label="pole angle")
  plt.xlabel("t")
  plt.legend()
  plt.show()

  # plt.plot(all_obs[:T_term, 1], all_obs[:T_term, 3])
  # plt.scatter(all_obs[0, 1], all_obs[0, 3], label="start point", zorder = 2)
  # plt.scatter(all_obs[T_term, 1], all_obs[T_term, 3], label="end point", zorder = 2)
  # plt.xlabel("Angle pole")
  # plt.ylabel("Angular velocity pole")
  # plt.title("Phase portrait t=1000")
  # plt.legend()
  # plt.show()

def compute_and_visualize(obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, dones) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, dones)

### Kozax

In [ ]:
brax_env = BraxGymnaxWrapper('inverted_pendulum', 'generalized')
policy = lambda obs: jnp.array([obs[1] + obs[2]**2])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000.]


### Modi

In [ ]:
brax_env = BraxGymnaxWrapper('inverted_pendulum', 'generalized')
policy = lambda obs: jnp.array([obs[1] + obs[2]**2])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000.]


In [ ]:
brax_env = BraxGymnaxWrapper('inverted_pendulum', 'generalized')
policy = lambda obs: jnp.array([2*obs[1] + obs[2] + obs[3]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000.]


In [ ]:
brax_env = BraxGymnaxWrapper('inverted_pendulum', 'generalized')
policy = lambda obs: jnp.array([2*obs[1] + obs[2] + obs[3]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000.]


### CGPAX

In [ ]:
brax_env = BraxGymnaxWrapper('inverted_pendulum', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([((0.1*obs[2]) + obs[1] + obs[3])]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000.]


In [ ]:
brax_env = BraxGymnaxWrapper('inverted_pendulum', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([((obs[0]*(obs[3] + obs[2])) + obs[1])]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000.]


In [ ]:
brax_env = BraxGymnaxWrapper('inverted_pendulum', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([((obs[3] + obs[2]) + (obs[1] + obs[1]))]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000. 1000.]


## Inverted double pendulum

Action Space: [-1, 1], where action represents the numerical force applied to the cart

Observation Space: $ℝ^{4}$

0.   car_position
1.   sin_angle_cart_first_pole
2.   sin_angle_two_poles
3.   cos_angle_cart_first_pole
4.   cos_angle_two_poles
5.   car_velocity
6.   angular_velocity_cart_first_pole
7.   angular_velocity_two_poles

Reward: reward = alive_bonus - distance_penalty - velocity_penalty.

  - *alive_bonus*: +10 for each timestep that the pole is upright
  - *distance_penalty*: movement of tip of the second pendulum, calculated as $0.01 x_{pole2-tip}^2 + (y_{pole2-tip}-2)^2$.
  - *velocity_penalty*: Negative reward to penalize fast movement.
  $10^{-3} \omega_1 + 5 \times 10^{-3} \omega_2$,
  where $\omega_1, \omega_2$ are the angular velocities of the hinges.

The episode terminates when episode duration reaches 1000 timestep or the y_coordinate of the tip of the second pole $\leq 1$.

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("inverted_double_pendulum", 'generalized')
T = 1000

/usr/local/lib/python3.12/dist-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        done = done | done_new.astype(bool)
        reward = jnp.where(done, 0.0, reward)

        return (obs, env_state, done), (obs, reward, action, done)

    return _step

def visualize_trajectory(all_obs, treward, actions, dones):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  all_dones = jnp.array(dones)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  reward = jnp.sum(all_rewards)
  all_obs = jnp.array(all_obs)
  plot_list = []

  cart_y = 0.0
  pole_length = 0.5

  for i in range(0, T_term, 50):
    print(
            f"Timestep {i} : "
            f"Cart position: {all_obs[i, 0]} | "
            f"Cart velocity: {all_obs[i, 5]} | "
            f"Angle cart-first pole: {jnp.atan2(all_obs[i, 1], all_obs[i, 3])} |"
            f"Angle first pole-second pole: {jnp.atan2(all_obs[i, 2], all_obs[i, 4])} |"
        )
    x = all_obs[i, 0]
    sin_angle_1 = all_obs[i, 1]
    cos_angle_1 = all_obs[i, 3]
    sin_angle_2 = all_obs[i, 2]
    cos_angle_2 = all_obs[i, 4]

    # Pole1 tip position
    joint_x = x + pole_length * sin_angle_1
    joint_y = cart_y + pole_length * cos_angle_1

    # Pole1 tip position
    tip_x = joint_x + pole_length * sin_angle_2
    tip_y = joint_y + pole_length * cos_angle_2

    plot_list.append([i, x, joint_x, joint_y, tip_x, tip_y])


  print(
          f"Timestep {T_term} : "
          f"Cart position: {all_obs[T_term, 0]} | "
          f"Cart velocity: {all_obs[T_term, 5]} | "
          f"Angle cart-first pole: {jnp.atan2(all_obs[T_term, 1], all_obs[T_term, 3])} |"
          f"Angle first pole-second pole: {jnp.atan2(all_obs[T_term, 2], all_obs[T_term, 4])} |"
      )
  x = all_obs[i, 0]
  sin_angle_1 = all_obs[T_term, 1]
  cos_angle_1 = all_obs[T_term, 3]
  sin_angle_2 = all_obs[T_term, 2]
  cos_angle_2 = all_obs[T_term, 4]

  # Pole1 tip position
  joint_x = x + pole_length * sin_angle_1
  joint_y = cart_y + pole_length * cos_angle_1

  # Pole1 tip position
  tip_x = joint_x + pole_length * sin_angle_2
  tip_y = joint_y + pole_length * cos_angle_2

  plot_list.append([T_term, x, joint_x, joint_y, tip_x, tip_y])

  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  fig, axs = plt.subplots(nrows=1, ncols=len(plot_list), figsize=(len(plot_list), 6))
  fig.suptitle("State evolution over time")
  for i in range(len(plot_list)):
    t, x, joint_x, joint_y, tip_x, tip_y = plot_list[i]
    axs[i].scatter(x, cart_y, c="black")
    axs[i].plot([x, joint_x], [cart_y, joint_y])
    axs[i].scatter(joint_x, joint_y, c="black")
    axs[i].plot([joint_x, tip_x], [joint_y, tip_y])
    axs[i].scatter(tip_x, tip_y, c="red")
    axs[i].set_title("t="+ str(t))
    axs[i].set_xlim(-2, 2)
    axs[i].set_ylim(-0.1, 1.0)
    axs[i].set_xlabel("x")
    if i == 0:
      axs[i].set_ylabel("y")
  plt.tight_layout()
  plt.show()

  print("Total reward:", reward)

  plt.plot(all_rewards[:T_term], label="reward")
  plt.plot(all_actions[:T_term], label="actions")
  plt.plot(all_obs[:T_term, 0], label="cart position")
  plt.plot(all_obs[:T_term, 5], label="cart velocity")
  plt.xlabel("t")
  plt.legend()
  plt.show()

def compute_and_visualize(key, obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, dones) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, dones)

### Kozax

In [ ]:
brax_env = BraxGymnaxWrapper('inverted_double_pendulum', 'generalized')
policy = lambda obs: jnp.array([obs[2]*(obs[4] + 1.06)*((obs[0] + obs[7])*(-obs[0] + obs[2] - (obs[3] + 2.18)*(obs[6] + 4*obs[7])) - 0.592) - obs[2]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[9359.226 9359.785 9359.732 9359.439 9359.197 9358.28  9359.703 9359.775
 9359.457 9359.726]


In [ ]:
brax_env = BraxGymnaxWrapper('inverted_double_pendulum', 'generalized')
policy = lambda obs: jnp.array([obs[1]*(2*obs[0] + obs[2] + obs[7])*(obs[0] + obs[2] - obs[5] + 3*obs[7]) - 2*obs[2]])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[9359.317 9359.586 9359.464 9359.478 9359.302 9359.212 9359.303 9359.653
 9359.496 9359.299]


In [ ]:
brax_env = BraxGymnaxWrapper('inverted_double_pendulum', 'generalized')
policy = lambda obs: jnp.array([-2*obs[2] + obs[7]*(1.71*obs[0] + obs[7])*(1.71*obs[0]*(2*obs[2] + obs[7]) + obs[3])*jnp.sin(obs[2] + obs[5] + obs[6])])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[9359.68  9359.912 9359.889 9359.803 9359.662 9359.478 9359.818 9359.9
 9359.682 9359.914]


### Modi

In [ ]:
brax_env = BraxGymnaxWrapper('inverted_double_pendulum', 'generalized')
policy = lambda obs: jnp.array([-obs[2]*(obs[0]**2 + obs[3] + 1.15) + jnp.sin(17.8*obs[2]*obs[7]*(2*obs[1] + obs[6]))])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[9358.526 9359.78  9359.69  9358.725 9358.593 9357.107 9359.534 9359.739
 9358.797 9359.789]


In [ ]:
brax_env = BraxGymnaxWrapper('inverted_double_pendulum', 'generalized')
policy = lambda obs: jnp.array([-2*obs[2] + 2*obs[7]**2*(-obs[0] - obs[7] - jnp.sin(-obs[2] + 3*obs[6] + 3*obs[7]))])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[9359.916 9359.973 9359.973 9359.91  9359.929 9359.84  9359.963 9359.969
 9359.953 9359.975]


In [ ]:
brax_env = BraxGymnaxWrapper('inverted_double_pendulum', 'generalized')
policy = lambda obs: jnp.array([1.14*obs[2]*((obs[0] + obs[7])*(obs[1] - obs[3]*(obs[2] + 5*obs[7])) - 1.76)])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[9358.643  9359.641  9359.482  9359.244  9358.5625 9355.811  9359.377
 9359.652  9358.572  9359.457 ]


### CGPAX

In [ ]:
brax_env = BraxGymnaxWrapper('inverted_double_pendulum', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([(obs[2]*(((obs[1]*obs[7]) - obs[4]) - jnp.sin(obs[3])))]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[9352.57   9359.363  9358.446  9357.552  9355.865  6592.1226 9358.223
 9359.582  9353.12   9358.161 ]


In [ ]:
brax_env = BraxGymnaxWrapper('inverted_double_pendulum', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([((((obs[6]*(obs[7]*obs[7])) - obs[2]) - obs[2]) - obs[2])]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[9356.605  9359.828  9359.461  9357.326  7894.3066 6481.136  9359.284
 9359.498  9358.141  9359.76  ]


In [ ]:
brax_env = BraxGymnaxWrapper('inverted_double_pendulum', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([(((((obs[7]*obs[6]) - jnp.cos(obs[2])) + (obs[3] + ((obs[7]*obs[6]) - jnp.cos(obs[2]))))*obs[2]) - obs[2])]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[9353.358  9358.992  9358.357  9357.205  9355.57   7442.9575 9358.171
 9359.091  9354.318  9358.29  ]


## Reacher

Action space: $[-1, 1]^2$, where the first action is the torque applied at the first hinge (connecting the link to the point of fixture), and the second action the torque applied at the second hinge (connecting the two links)

Observation space: $ℝ^{11}$

0.   cos(joint0)
1.   cos(joint1)
2.   sin(joint0)
3.   sin(joint1)
4.   target_x
5.   target_y
6.   ang_vel joint 0
7.   ang_vel joint 1
8.   diff x-value
9.   diff y-value
10.  (diff z-value)

The reward consists of two parts:

  - *reward_dist*: distance between *fingertip* of the reacher and the target,  with a more negative value assigned when *fingertip* is further away.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as the negative squared Euclidean norm of the action.


   The episode terminates when episode duration reaches a 1000 timesteps


### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("reacher", 'generalized')
T = 1000
#env = EpisodeWrapper(envs.get_environment("reacher"), episode_length=T, action_repeat=1)

In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, new_state, reward, done, _ = env.step(
            env_state,
            action,
            None
        )

        return (obs, new_state, done.astype(bool)), (obs, reward, action, done)

    return _step

def visualize_trajectory(all_obs, treward, actions):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  all_obs = jnp.array(all_obs)
  all_list = []
  plot_list = []

  for i in range(1000):
    theta1 = jnp.arctan2(all_obs[i, 2], all_obs[i, 0])
    theta2 = jnp.arctan2(all_obs[i, 3], all_obs[i, 1])

    x1 = 0.1 * jnp.cos(theta1)
    y1 = 0.1 * jnp.sin(theta1)

    x2 = x1 + 0.1 * jnp.cos(theta1 + theta2)
    y2 = y1 + 0.1 * jnp.sin(theta1 + theta2)

    all_list.append([x2, y2])

    if i % 50 == 0:

      print(
              f"Timestep {i} : "
              f"x-coordinate tip: {x2} | "
              f"y-coordinate tip: {y2} | "
              f"Angular velocity joint 1: {all_obs[i, 6]} | "
              f"Angular velocity joint 2: {all_obs[i, 7]} | "
          )

      plot_list.append([i, x1, y1, x2, y2])

  nr_of_rows = 4
  nr_of_cols = 5

  fig, axs = plt.subplots(nrows=nr_of_rows, ncols=nr_of_cols, figsize=(12, 12))
  fig.suptitle("State evolution over time")
  for i in range(len(plot_list)):
    c = i % nr_of_cols
    r = int(i / nr_of_cols)
    t, x1, y1, x2, y2 = plot_list[i]
    axs[r, c].scatter(0,0, c = "black", label = "middle point")
    axs[r, c].scatter(x2, y2, c="blue", label = "arm tip")
    axs[r, c].plot([0, x1], [0, y1])
    axs[r, c].plot([x1, x2], [y1, y2])
    axs[r, c].scatter(all_obs[t, 4], all_obs[t, 5], c = "red", label = "target location")

    if i != 0:
      x2s_y2s = jnp.array(all_list[t-50:t])
      x2s = x2s_y2s[:, 0]
      y2s = x2s_y2s[:, 1]
      axs[r, c].plot(x2s, y2s, alpha = 0.4, label = "tip trajectory")

    axs[r, c].set_title("t="+ str(t))
    axs[r, c].set_xlim(-0.3, 0.3)
    axs[r, c].set_ylim(-0.3, 0.3)
    if r == 0 and c == nr_of_cols-1:
      axs[r, c].legend(loc='best', fontsize = 6)
    if i >= (nr_of_rows-1) * nr_of_cols:
      axs[r, c].set_xlabel("x")
    if c == 0:
      axs[r, c].set_ylabel("y")

  plt.tight_layout()
  plt.show()

  print("Total reward:", reward)

  plt.plot(all_obs[:,8], alpha = 0.4, label='x-value diff')
  plt.plot(all_obs[:,9], alpha = 0.4, label='y-value diff')
  plt.plot(jnp.add(jnp.abs(all_obs[:, 8]), jnp.abs(all_obs[:, 9])), label="total value diff")
  plt.plot(all_rewards, label="reward")
  plt.plot(all_actions, label="actions")
  plt.legend()
  plt.show()

def compute_and_visualize(obs, env_state, policy, T):

  (_, _, _), (all_obs, treward, actions, dones) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions)

### Kozax

In [ ]:
brax_env = BraxGymnaxWrapper('reacher', 'generalized')
policy = lambda obs: jnp.array([
    obs[8]*(obs[0] + obs[7])*(obs[4] - obs[8])*(-0.653*obs[4] + obs[9] - 0.653*(obs[4] - obs[8])*(-0.653*obs[4]*obs[9] - 0.653*obs[4] + obs[9])),
    obs[5]*(5.69*obs[2]*obs[9] + obs[4]*obs[8] + obs[8])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[-122.18803   -39.341423  -39.4693   -129.34683  -228.1581    -73.61695
  -66.75958   -29.879534  -52.23596  -138.57777 ]


In [ ]:
brax_env = BraxGymnaxWrapper('reacher', 'generalized')
policy = lambda obs: jnp.array([
    obs[5]*obs[8] + obs[6]*(obs[5]*obs[6] + obs[5]) + obs[8]*(0.000939*obs[6]*obs[8] + 0.000939*obs[7] - 0.0112),
    obs[10]*obs[2] + obs[2]*obs[9]*(obs[2] + obs[9]) + obs[5]*(obs[0]*obs[8]*(2*obs[0] + obs[8]) + obs[8] + obs[9])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[-44.517166 -54.970894 -57.37097  -40.8796   -13.739962 -12.224989
 -12.050418 -48.131725 -14.703627 -45.299652]


In [ ]:
brax_env = BraxGymnaxWrapper('reacher', 'generalized')
policy = lambda obs: jnp.array([
    obs[10]**2*obs[2]*(-obs[0]*(-obs[10] - 1.38) + obs[5]) - jnp.sin(obs[8]*(obs[4] + obs[9])),
    obs[2]*obs[9] + jnp.sin(jnp.sin(obs[8]*(-0.437*obs[2] + jnp.sin(obs[5]*obs[8]) + jnp.sin(2*obs[0] + obs[10] + obs[5]))))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[-15.646914 -11.650794 -10.113796 -17.040308 -51.033974  -8.081129
 -11.415045 -10.876755 -13.559533 -19.182598]


### Modi

In [ ]:
brax_env = BraxGymnaxWrapper('reacher', 'generalized')
policy = lambda obs: jnp.array([
    obs[4]*obs[8]*(-obs[10]*obs[5] + obs[5]*obs[8]) + obs[5]*obs[8],
    obs[10]*obs[5] + obs[4]*obs[8]*(-obs[10]*obs[5] + obs[5]*obs[8])
    + obs[5]**3*(2*obs[1] - obs[4]*obs[8]*(-obs[10]*obs[5] + obs[5]*obs[8]) - obs[5] - obs[9] + jnp.sin(jnp.sin(obs[2])))
    + obs[5]**2*(2*obs[1] - obs[4]*obs[8]*(-obs[10]*obs[5] + obs[5]*obs[8]) - obs[5] - obs[9] + jnp.sin(jnp.sin(obs[2])))
    + obs[5]*(2*obs[1] - obs[4]*obs[8]*(-obs[10]*obs[5] + obs[5]*obs[8]) - obs[5] - obs[9] + jnp.sin(jnp.sin(obs[2])))
    + obs[5] + obs[9]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[ -32.974102 -157.00455  -174.26735   -22.220722  -93.41994   -13.100274
  -14.925985 -202.68611   -42.621777  -19.290703]


In [ ]:
brax_env = BraxGymnaxWrapper('reacher', 'generalized')
policy = lambda obs: jnp.array([
    obs[2]*obs[3]*obs[9],
    obs[2]*obs[3]*obs[9]
    - obs[8]*(-obs[10] + obs[8]*jnp.sin(obs[5] - 0.253) - obs[9] - jnp.sin(obs[2] + jnp.sin(obs[5] - 0.253)))*jnp.sin(0.253*obs[0])
    - obs[8]*jnp.sin(0.253*obs[0])
    + obs[8]*jnp.sin(obs[5] - 0.253)
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[ -24.732977  -140.1647      -7.3620453  -20.566296  -198.91632
  -56.344284   -55.23688   -100.42915    -58.55442    -97.43886  ]


In [ ]:
brax_env = BraxGymnaxWrapper('reacher', 'generalized')
policy = lambda obs: jnp.array([
    jnp.sin(0.586*obs[5]*obs[8]) + jnp.sin(obs[8]*obs[9]),
    obs[0]*obs[9]*jnp.sin(obs[8]*obs[9]) + obs[0]*obs[9]
    + 0.586*obs[1]*obs[2]*obs[8]
    + 2*obs[1]*(obs[0]*obs[9]*jnp.sin(obs[8]*obs[9]) + obs[5] + jnp.sin(0.586*obs[5]*obs[8]))
    + obs[5]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[-34.02261   -91.97061   -56.50609   -29.756634  -89.94681   -25.401257
 -31.747074  -90.70827   -34.365555  -15.5869255]


### CGPAX

In [ ]:
brax_env = BraxGymnaxWrapper('reacher', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([((obs[9]*((obs[8] + obs[6]) - 0.1)) - (0.1*obs[7])), (obs[8] + obs[6])]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[ -11.035684   -8.032543   -6.649865  -17.572933 -204.86156   -91.60234
  -58.990433   -8.2789   -131.2038    -34.697296]


In [ ]:
brax_env = BraxGymnaxWrapper('reacher', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([(obs[9]*((obs[10] - obs[10]) - jnp.sin(0.1))), obs[8]]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[  -8.36762  -156.42293  -186.84341   -16.191692 -186.60527   -83.84896
  -65.064026 -166.7888   -127.940125  -28.766724]


In [ ]:
brax_env = BraxGymnaxWrapper('reacher', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([((obs[8]*jnp.sin(obs[3] + obs[2])) - jnp.sin(obs[7] + obs[9])), obs[8]]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[ -12.18174  -158.4203     -8.789865  -19.91475  -195.57431   -80.80366
  -77.29628   -27.56998  -122.19178   -35.739536]


## Swimmer

Action space: $[-1, 1]^2$, where the first action is the torque applied on the first rotor, and the second action the torque applied on the second rotor.

Observation space: $ℝ^{8}$

0.   angle_front_tip
1.   angle_second_rotor
2.   angle_third_rotor
3.   velocity_tip_x-axis
4.   velocity_tip_y-axis
5.   angular_velocity_front_tip
6.   angular_velocity_second_rotor
7.   angular_velocity_third_rotor

The reward consists of two parts:

  - *reward_fwd*: A reward of moving forward which is measured as (x-coordinate before action - x-coordinate after action) / dt.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as *-coefficient x sum(action<sup>2</sup>)*


   The episode terminates when episode duration reaches a 1000 timesteps

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("swimmer", 'generalized')
T = 1000

In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        return (obs, env_state, done.astype(bool)), (obs, reward, action, env_state.pipeline_state.x.pos)

    return _step

def visualize_trajectory(all_obs, treward, actions, all_pos):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  all_obs = jnp.array(all_obs)
  all_pos = jnp.array(all_pos)
  plot_list = []

  for i in range(0, 1000, 50):
    print(
            f"Timestep {i} : "
            f"x-coordinate tip: {all_pos[i, -1, 0]} | "
            f"y-coordinate tip: {all_pos[i, -1, 1]} | "
        )
    xy = all_pos[i]
    plot_list.append([i, xy[:, 0], xy[:, 1]])

  nr_of_rows = 4
  nr_of_cols = 5

  fig, axs = plt.subplots(nrows=nr_of_rows, ncols=nr_of_cols, figsize=(12, 12))
  fig.suptitle("State evolution over time")
  for i in range(len(plot_list)):
    t, x, y = plot_list[i]
    c = i % nr_of_cols
    r = int(i / nr_of_cols)
    axs[r, c].plot(x, y)
    axs[r, c].scatter(x, y)
    x_till_t = all_pos[:t, -1, 0]
    y_till_t = all_pos[:t, -1, 1]
    axs[r, c].plot(x_till_t, y_till_t, label = "tip trajectory")
    axs[r, c].set_title("State at t="+ str(t))
    axs[r, c].set_xlim(-2,15)
    axs[r, c].set_ylim(-2,2)
    if i > (nr_of_rows-1) * nr_of_cols:
      axs[r, c].set_xlabel("x")
    if c == 0:
      axs[r, c].set_ylabel("y")
  plt.tight_layout()
  plt.show()

  print("Total reward:", reward)

  x = all_pos[:, -1, 0]
  y = all_pos[:, -1, 1]
  plt.plot(jnp.arange(1000), x, label = "x")
  plt.plot(jnp.arange(1000), y, label = "y")
  plt.xlabel("t")
  plt.ylabel("value")
  plt.legend()
  plt.show()

  plt.plot(x, y, label = "tip trajectory")
  plt.xlabel("x")
  plt.ylabel("y")
  plt.legend()
  plt.show()

  plt.plot(all_rewards, label="reward")
  plt.plot(all_actions, label="actions")
  plt.legend()
  plt.show()

def compute_and_visualize(obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, all_pos) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, all_pos)

### Kozax

In [ ]:
brax_env = BraxGymnaxWrapper('swimmer', 'generalized')
policy = lambda obs: jnp.array([
    -0.131*obs[0]*(-0.131*obs[0]**2*jnp.sin(obs[6] + obs[7]) + obs[0] + obs[2] + 0.822) + 2*obs[0] + obs[6],
    -obs[2] - 0.588*obs[4] - obs[5] + 0.0295
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[365.8954  367.49225 366.32227 367.89185 369.08704 368.97913 366.61792
 367.42603 364.2873  368.55783]


In [ ]:
brax_env = BraxGymnaxWrapper('swimmer', 'generalized')
policy = lambda obs: jnp.array([
    2*obs[0] + obs[4] + obs[5] + obs[6] + jnp.sin(obs[4]) + jnp.cos(obs[6]),
    (obs[0]*jnp.cos(obs[1]**3*obs[4]**2) + obs[1] + obs[4])*jnp.cos(jnp.cos(obs[0]))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[364.61597 366.54953 367.5144  367.58905 367.25696 366.59427 364.85065
 367.05438 365.61676 365.56146]


In [ ]:
brax_env = BraxGymnaxWrapper('swimmer', 'generalized')
policy = lambda obs: jnp.array([
    -2*obs[2] - jnp.sin(obs[2] - jnp.sin(obs[3]*(obs[1] + obs[4]) + jnp.sin(obs[0] + obs[4] - jnp.sin(jnp.sin(obs[2] + jnp.sin(obs[2] - jnp.sin(jnp.sin(obs[1] + obs[4])))))))),
    obs[1] + obs[2] + obs[6]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[369.04218 370.7457  370.65674 370.61206 371.89758 369.17664 369.33655
 368.25525 368.76007 370.85425]


### Modi

In [ ]:
brax_env = BraxGymnaxWrapper('swimmer', 'generalized')
policy = lambda obs: jnp.array([
    4*obs[0] + obs[1] + obs[3]*(-0.908*obs[0] - 0.908*obs[4] - 0.432*obs[5]) + obs[6],
    0.908*obs[0] + obs[1] + obs[3]*(-0.908*obs[0] - 0.908*obs[4] - 0.432*obs[5]) + 0.908*obs[4]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[364.36243 366.76202 366.92563 367.32538 366.08167 367.24072 364.62628
 366.2287  365.4749  363.1192 ]


In [ ]:
brax_env = BraxGymnaxWrapper('swimmer', 'generalized')
policy = lambda obs: jnp.array([
    2*obs[0] - 0.00641*obs[5] + obs[6],
    2*obs[0] + 3*obs[1] - 4*obs[2] + 0.0476*obs[3]*obs[7] + 0.997*obs[5] + obs[6]
    + 3*jnp.sin(obs[1])
    + jnp.cos(2*obs[0] + obs[1] - 2*obs[2] + 0.0476*obs[3]*obs[7] + 0.997*obs[5] + obs[6] + jnp.sin(obs[1]) + 0.488)
    + 0.488
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[365.49985 367.12408 367.06207 368.4285  368.64355 367.22406 366.10062
 367.36215 366.41602 367.53326]


In [ ]:
brax_env = BraxGymnaxWrapper('swimmer', 'generalized')
policy = lambda obs: jnp.array([
    2*obs[0] + 0.103*obs[1] - 1.36*obs[2] + obs[4] + 0.103*obs[6]
    + 0.103*jnp.sin(obs[1])
    + 0.103*jnp.sin(obs[1] + obs[6])
    - 0.221*jnp.sin(jnp.sin(obs[1])),
    0.103*obs[1] - 2.72*obs[2] + obs[4] + 0.103*obs[6]
    + 2.1*jnp.sin(obs[1])
    + 0.103*jnp.sin(obs[1] + obs[6])
    + 0.779*jnp.sin(jnp.sin(obs[1]))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[364.74258 367.26385 366.94568 367.594   367.85394 367.57642 365.0114
 366.71948 361.0282  367.13416]


### CGPAX

In [ ]:
brax_env = BraxGymnaxWrapper('swimmer', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([((obs[6] - obs[2])*jnp.cos(obs[3]*jnp.cos(obs[7]))), (obs[4] - obs[5])]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[350.67618 359.71487 360.0426  370.66602 366.38965 370.38928 357.47217
 366.41058 367.93625 366.31366]


In [ ]:
brax_env = BraxGymnaxWrapper('swimmer', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([(obs[6] + jnp.sin(jnp.sin(obs[5]))), ((obs[4] + obs[1]) - obs[5])]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[364.09424 364.36157 363.3308  365.70435 364.25543 364.516   359.8185
 364.04718 360.91595 360.93628]


In [ ]:
brax_env = BraxGymnaxWrapper('swimmer', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([(obs[0] + obs[6]), (obs[4] + ((obs[3]*obs[1]) + (obs[3]*obs[1])))]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[361.2046  364.4168  364.81604 364.75574 354.20346 366.26678 364.65796
 362.03864 360.73914 350.1306 ]


## Hopper

Action space: $[-1, 1]^3$, where the first action is the torque applied on the thigh rotor, and the second action the torque applied on the leg rotor, and the thrid action the torque applied to the foot rotor.

Observation space: $ℝ^{11}$

0.   z-coordinate of the top
1.   angle_top
2.   angle_thigh_joint
3.   angle_leg_joint
4.   angle_foot_joint
5.   velocity_x-coordinate_top
6.   velocity_z-coordinate_top
7.   angular_velocity_angle_top
8.   angular_velocity_thigh_hinge
9.   angular_velocity_leg_hinge
10.  angular_velocity_foot_hinge

The reward consists of three parts:

  - *reward_healthy*: +1 reward for every timestep that the hopper is alive.

  - *reward_fwd*: A reward of moving forward which is measured as (x-coordinate before action - x-coordinate after action) / dt.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as *-coefficient x sum(action<sup>2</sup>)*


   The episode terminates when episode duration reaches a 1000 timesteps, the height of the hopper becomes less than 0.7 metres, or the absolute value of the angle of the thigh joint is less than 0.2 radians

### Visualize

In [ ]:
env = BraxGymnaxWrapper("hopper", 'generalized')
T = 1000

/usr/local/lib/python3.12/dist-packages/brax/io/mjcf.py:480: UserWarning: Brax System, piplines and environments are not actively being maintained. Please see MJX for a well maintained JAX-based physics engine: https://github.com/google-deepmind/mujoco/tree/main/mjx. For a host of environments that use MJX, see: https://github.com/google-deepmind/mujoco_playground.
  warnings.warn(


In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        done = done | done_new.astype(bool)
        reward = jnp.where(done, 0.0, reward)

        return (obs, env_state, done), (obs, reward, action, done)#, env_state.pipeline_state.x.pos, env_state.pipeline_state.q)

    return _step

def visualize_trajectory(all_obs, treward, actions, dones, all_pos, all_angles):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  all_dones = jnp.array(dones)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  all_obs = jnp.array(all_obs)
  all_pos = jnp.array(all_pos)
  all_angles = jnp.array(all_angles)
  plot_list = []

  for i in range(0, T_term, 50):

    print(
            f"Timestep {i} : "
            f"Height: {all_obs[i, 0]} | "
            f"x-coordinate thigh: {all_pos[i][2, 0]} | "
            f"Thigh angle (rad): {all_angles[i, 2]} | "
        )
    xz = all_pos[i][:, [0, 2]]
    plot_list.append([i, xz])

  print(
          f"Timestep {T_term} : "
          f"Height: {all_obs[T_term, 0]} | "
          f"x-coordinate thigh: {all_pos[T_term][2, 0]} | "
          f"Thigh angle (rad): {all_angles[T_term, 2]} | "
        )
  xz = all_pos[T_term][:, [0, 2]]
  plot_list.append([T_term, xz])
  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  # if T_term < 1000:
  #   print("Height:", all_obs[T_term, 0], ", Angle (rad)", all_angles[T_term, 2])
  #   xz = all_pos[T_term][:, [0, 2]]
  #   plt.plot(xz[:, 0], xz[:, 1], "-o")
  #   plt.title("State at t="+ str(T_term))
  #   plt.xlim(-0.25, 0.25)
  #   plt.ylim(-0.2, 1.5)
  #   plt.show()

  #   plt.show()
  #   print("Terminated at timestep:", T_term)
  # else:
  #   print("Height:", all_obs[1000, 0], ", Angle (rad)", all_angles[1000, 2])
  #   xz = all_pos[1000][:, [0, 2]]
  #   plt.plot(xz[:, 0], xz[:, 1], "-o")
  #   plt.title("State at t="+ str(1000))
  #   plt.xlim(-0.25, 0.25)
  #   plt.ylim(-0.2, 1.5)
  #   plt.show()
  #   print("No termination (ran full 1000 steps)")

  fig, axs = plt.subplots(nrows=1, ncols=len(plot_list), figsize=(max(6, len(plot_list)), 6))
  fig.suptitle("State evolution over time")
  for i in range(len(plot_list)):
    t, xz = plot_list[i]
    axs[i].plot(xz[:, 0], xz[:, 1], "-o")
    axs[i].set_title("t="+ str(t))
    axs[i].set_xlim(-0.25, 2)
    axs[i].set_ylim(-0.2, 1.5)
    axs[i].set_xlabel("x")
    if i == 0:
      axs[i].set_ylabel("z")
  plt.tight_layout()
  plt.show()

  print("Total reward:", reward)

  plt.plot(all_rewards[:T_term], label="reward")
  plt.plot(all_actions[:T_term], label=["action thigh", "action leg", "action foot"])
  plt.xlabel("t")
  plt.ylabel("value")
  plt.legend()
  plt.show()

def compute_and_visualize(obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, all_dones, all_pos, all_angles) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, all_dones, all_pos, all_angles)

### Kozax

In [ ]:
brax_env = BraxGymnaxWrapper('hopper', 'generalized')
policy = lambda obs: jnp.array([
    -0.568*obs[7],
    obs[1]*(obs[1]*obs[2]*obs[3]*(obs[3]*obs[7] + obs[5])*(obs[0] + obs[3]**2 + obs[3]) + obs[3]),
    obs[2]*obs[5]*(obs[4] + obs[9])*jnp.cos(obs[2]*obs[8] - obs[5] + obs[6]) + obs[2]*obs[8] - obs[5] + obs[7]*obs[8]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1198.587  1237.1372 1196.4631 1235.235  1249.8798 1255.2429 1283.3235
 1267.4395 1277.6423 1262.4597]


In [ ]:
brax_env = BraxGymnaxWrapper('hopper', 'generalized')
policy = lambda obs: jnp.array([
    0.123*jnp.cos(obs[6]),
    obs[7]*(-obs[0] + obs[3] + obs[6]*(-obs[8]*(obs[1] + 2*obs[4] + 0.471) + 0.471)*(-obs[0] + obs[3]*(obs[3] + obs[6]) + obs[3] + obs[5])),
    obs[0]*obs[2]*(0.665*obs[0]*obs[4]*obs[6]*(obs[4] + obs[9]) + obs[2]) + obs[0]*obs[2] + obs[0]*obs[4]*(obs[4] + obs[9]) - obs[5]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1178.8473 1332.1698 1344.694  1222.6656 1168.7726 1404.8796 1341.7614
 1202.6556 1354.665  1285.164 ]


In [ ]:
brax_env = BraxGymnaxWrapper('hopper', 'generalized')
policy = lambda obs: jnp.array([
    jnp.cos(obs[1]),
    (obs[0] - 0.846*jnp.cos(obs[9])*jnp.cos(obs[8] + obs[9] + jnp.sin(obs[4]) - jnp.cos(obs[6] + obs[8])))*(-obs[7] + obs[8]),
    -2.23*obs[5] + jnp.cos(obs[4]*(-obs[10] + obs[4]*(-obs[10] - 2.23*obs[5]*obs[6]*(obs[1] + obs[3]*obs[5] - obs[6]))))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1434.0886 1364.1893 1285.3658 1485.8643 1353.601  1468.7781 1359.5776
 1345.9603 1397.3347 1389.9001]


### Modi

In [ ]:
brax_env = BraxGymnaxWrapper('hopper', 'generalized')
policy = lambda obs: jnp.array([
    obs[0]*(obs[0] + obs[1]*obs[3] + obs[7]*(obs[1] + obs[10]*obs[3]*obs[8]) + obs[8]**2*(obs[1]*obs[10] - obs[10]*obs[3]))
    + obs[7]*(obs[1] + obs[10]*obs[3]*obs[8])
    + obs[8]**2*(obs[1]*obs[10] - obs[10]*obs[3]),
    obs[1]*obs[3] + obs[7]*(obs[1] + obs[10]*obs[3]*obs[8]),
    obs[1]*obs[10] + 2*obs[10]*obs[3] + obs[8]**2
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1111.5277 1033.5074 1029.3164 1031.2168 1032.0062 1080.5305 1036.6115
 1027.6244 1095.0873 1100.9465]


In [ ]:
brax_env = BraxGymnaxWrapper('hopper', 'generalized')
policy = lambda obs: jnp.array([
    obs[0]*obs[2]*obs[6]**2 - 0.151*obs[6],
    -obs[7] + obs[9]*(-0.151*obs[6]*(obs[6] + obs[9]*jnp.cos(obs[7] + jnp.cos(jnp.cos(obs[0]*obs[2]*obs[6]**2*obs[9])))) + obs[7] - 0.151),
    obs[9]*jnp.cos(obs[7] + jnp.cos(jnp.cos(obs[0]*obs[2]*obs[6]**2*obs[9])))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1170.7567 1163.8503 1117.8496 1172.4641 1169.5288 1145.1571 1141.6036
 1160.2998 1126.6707 1167.1576]


In [ ]:
brax_env = BraxGymnaxWrapper('hopper', 'generalized')
policy = lambda obs: jnp.array([
    0.142*obs[0] - 2*obs[1] + obs[3],
    0.213*obs[0]
    + obs[1]*(obs[1] + obs[2])*(obs[1] + 0.0709*obs[3])*(0.00155*obs[0] - 0.0218*obs[1] + 0.0218*obs[3] - 0.0218*obs[5])
    + obs[1]*(obs[1] + 0.0709*obs[3])*(0.00155*obs[0] - 0.0218*obs[1] + 0.0218*obs[3] - 0.0218*obs[5])
    + obs[1]*(obs[1] + 0.0709*obs[3]),
    0.0725*obs[0]
    + obs[1]*(obs[1] + obs[2])*(obs[1] + 0.0709*obs[3])*(0.00155*obs[0] - 0.0218*obs[1] + 0.0218*obs[3] - 0.0218*obs[5])
    - 1.02*obs[1] + 1.02*obs[3] - 1.02*obs[5]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1196.4818 1216.8835 1196.296  1185.3173 1203.5005 1171.4755 1203.7477
 1226.34   1212.7119 1216.5133]


### CGPAX

In [ ]:
brax_env = BraxGymnaxWrapper('hopper', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([((jnp.sin(1) - 1)*obs[7]), ((1 - jnp.sin(1)) - obs[3]), (obs[9] - obs[5])]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1180.4431  1123.9026  1101.867    995.5169  1177.739   1018.33203
 1169.1917  1173.5074  1139.292   1159.5062 ]


In [ ]:
brax_env = BraxGymnaxWrapper('hopper', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([(jnp.cos(obs[6]) + obs[3]), jnp.cos(jnp.sin(obs[6])), (obs[4] - ((obs[6] + obs[7]) + obs[5]))]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[ 245.75212 1174.183   1172.3977   208.6787  1183.1611  1166.7006
 1182.0989  1163.5156  1165.5189  1174.8274 ]


In [ ]:
brax_env = BraxGymnaxWrapper('hopper', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([jnp.cos(obs[2] + obs[7]), jnp.sin(jnp.cos(obs[2] + obs[7])), (0.1 - ((obs[2] + obs[7]) + obs[6] + (obs[2] + obs[7])))]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[ 342.45352 1250.7745   390.88644  299.95026  430.491   1248.4395
 1246.533    256.99127 1246.1677  1251.9224 ]


## Walker2d

Action space: $[-1, 1]^6$, where the first action is the torque applied on the thigh rotor, the second action the torque applied on the leg rotor, the third action the torque applied to the foot rotor, the fourth action is the torque applied on the left thigh rotor, the fifth action the torque applied on the left leg rotor, and the sixth action the torque applied to the left foot rotor.

Observation space: $ℝ^{18}$

0. z-coordinate_top
1. angle_top
2. angle_thigh_joint
3. angle_leg_joint
4. angle_foot_joint
5. angle_left_thigh_joint
6. angle_left_leg_joing
7. angle_left_foot_joint
8. velocity_x-coordinate_top
9. velocity_z-coordinate_top
10. angular_velocity_top
11. angular_velocity_thigh_hinge
12. angular_velocity_leg_hinge
13. angular_velocity_foot_hinge
14. angular_velocity_thigh_hinge
15. angular_velocity_leg_hinge
16. angular_velocity_foot_hinge

The reward consists of three parts:

  - *reward_healthy*:
    +1 for each timepoint that the walker is alive
  - *reward_forward*:
    Reward of walking forward, measured as *(x-coordinate before action - x-coordinate after action) / dt*.
  - *reward_ctrl*:
    Negative reward for penalising the walker if it takes too large actions, measured as *-coefficient **x** sum(action<sup>2</sup>)*.

   The episode terminates when episode duration reaches a 1000 timesteps, when the height of the walker is not in the range [0.7, 2], and the absolute value of the angle is not in the range [-1, 1].

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("walker2d", 'generalized')
T = 1000

In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        return (obs, env_state, done), (obs, reward, action, done)#, env_state.pipeline_state.x.pos, env_state.pipeline_state.q)

    return _step

def visualize_trajectory(all_obs, treward, actions, all_dones, all_pos, all_angles):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  print(reward)
  all_obs = jnp.array(all_obs)
  all_pos = jnp.array(all_pos)
  all_angles = jnp.array(all_angles)
  plot_list = []

  for i in range(0, T_term, 50):

    print(
            f"Timestep {i} : "
            #f"Height: {all_obs[i, 0]} | "
            f"x-coordinate: {all_angles[i][0]} | "
            f"z-coordinate: {all_angles[i][2]} | "
            #f"Thigh angle (rad): {all_angles[i, 2]} | "
        )
    xz = all_pos[i][:, [0, 2]]
    plot_list.append([i, xz])

  print(
          f"Timestep {T_term} : "
          #f"Height: {all_obs[T_term, 0]} | "
          f"x-coordinate: {all_angles[i][0]} | "
          f"z-coordinate: {all_angles[i][2]} | "
          #f"Thigh angle (rad): {all_angles[T_term, 2]} | "
        )
  xz = all_pos[T_term][:, [0, 2]]
  plot_list.append([T_term, xz])
  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  # for i in range(0, 1000, 10):
  #   xy = all_pos[i][:, :2]

  #   plt.plot(xy[:, 0], xy[:, 1])
  #   plt.scatter(xy[:, 0], xy[:, 1])
  #   plt.title("State at t="+ str(i))
  #   plt.gca().set_aspect("equal")
  #   plt.show()

  # xy = all_pos[:, -1, :]
  # x = xy[:, 0]
  # y = xy[:, 1]
  # plt.plot(jnp.arange(1000), x, label = "x")
  # plt.plot(jnp.arange(1000), y, label = "y")
  # plt.legend()
  # plt.show()
  # plt.plot(x, y, label = "tip trajectory")
  # plt.legend()
  # plt.show()

  plt.plot(all_rewards, label="reward")
  plt.plot(all_actions, label="actions")
  plt.legend()
  plt.show()

def compute_and_visualize(key, obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, dones, all_pos, all_angles) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, dones, all_pos, all_angles)

### Kozax

In [ ]:
brax_env = BraxGymnaxWrapper('walker2d', 'generalized')
policy = lambda obs: jnp.array([
    1.56,
    -obs[1] - obs[10] - obs[14] + obs[15],
    obs[0]**2 + obs[3],
    jnp.cos(obs[8]),
    2*obs[0] + obs[3],
    obs[1] + jnp.cos(obs[1]**2)
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1207.8403 1190.7443 1170.0468 1216.1128 1191.132  1216.4344 1177.4072
 1206.277  1217.2681 1209.5336]


In [ ]:
brax_env = BraxGymnaxWrapper('walker2d', 'generalized')
policy = lambda obs: jnp.array([
    obs[6] + 2*jnp.cos(obs[3]),
    0.24*obs[0] + 0.233,
    jnp.cos(jnp.cos(jnp.cos(obs[3] + jnp.cos(obs[4])))),
    jnp.cos(obs[4] + 2*obs[9]),
    2*obs[1]*obs[6],
    2.64
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1313.7906  1354.3674  1340.5549  1339.8284  1321.164    278.54993
 1318.6687  1326.532   1366.69    1327.5879 ]


In [ ]:
brax_env = BraxGymnaxWrapper('walker2d', 'generalized')
policy = lambda obs: jnp.array([
    obs[7],
    obs[6]**2 + jnp.sin(obs[0]),
    jnp.cos(obs[6]),
    -2*obs[10] + obs[16],
    jnp.cos(obs[5] + obs[6]),
    obs[0] - obs[5] + jnp.sin(jnp.sin(obs[5]))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[ 921.5767  687.4643  852.0639 1079.3014  626.9593  867.3774  761.4415
  936.2865 1004.7622  959.9776]


### Modi

In [ ]:
brax_env = BraxGymnaxWrapper('walker2d', 'generalized')
policy = lambda obs: jnp.array([
    0,
    0,
    obs[3] + obs[5]*jnp.sin(obs[7]*obs[8]*jnp.sin(obs[5]*(obs[5] + obs[6])*(obs[1]*obs[5]*obs[6]*(obs[15] + obs[6]) + obs[11] - 1.74))),
    obs[1]*obs[5]*obs[6]*(obs[15] + obs[6]),
    obs[5]*(obs[5] + obs[6])*(obs[1]*obs[5]*obs[6]*(obs[15] + obs[6]) + obs[11] - 1.74)
    + obs[6]*(obs[15] + obs[6])
    + jnp.sin(obs[5]*(obs[5] + obs[6])*(obs[1]*obs[5]*obs[6]*(obs[15] + obs[6]) + obs[11] - 1.74))
    + jnp.sin(obs[7]*obs[8]*jnp.sin(obs[5]*(obs[5] + obs[6])*(obs[1]*obs[5]*obs[6]*(obs[15] + obs[6]) + obs[11] - 1.74))),
    obs[5]*jnp.sin(obs[7]*obs[8]*jnp.sin(obs[5]*(obs[5] + obs[6])*(obs[1]*obs[5]*obs[6]*(obs[15] + obs[6]) + obs[11] - 1.74)))
    + obs[7]*obs[8]*jnp.sin(obs[5]*(obs[5] + obs[6])*(obs[1]*obs[5]*obs[6]*(obs[15] + obs[6]) + obs[11] - 1.74))
    + obs[8]*jnp.sin(obs[5]*(obs[5] + obs[6])*(obs[1]*obs[5]*obs[6]*(obs[15] + obs[6]) + obs[11] - 1.74))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[568.2086  728.49365 522.9885  597.54785 529.9429  523.1466  489.54785
 639.9201  534.82104 580.3745 ]


In [ ]:
brax_env = BraxGymnaxWrapper('walker2d', 'generalized')
policy = lambda obs: jnp.array([
    0,
    0,
    0,
    obs[3] + 2*jnp.sin(obs[1]*obs[2]),
    obs[15]*obs[6]*jnp.cos(obs[5]*obs[8]),
    obs[5]*obs[8]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[  83.42742  883.70734  217.6939   685.3451  1574.7753   449.82715
  557.4971   578.3159   555.0335   586.95233]


In [ ]:
brax_env = BraxGymnaxWrapper('walker2d', 'generalized')
policy = lambda obs: jnp.array([
    0,
    0,
    0,
    obs[14]*obs[2]*obs[6]*(obs[15]*obs[2]*obs[6]**2 - obs[15]*jnp.sin(jnp.sin(0.517*obs[8])) - obs[15])*jnp.sin(obs[2]**2*obs[8])
    + obs[14]*obs[2]
    + obs[15]*obs[2]*obs[6]**2
    - 2*obs[15]*jnp.sin(jnp.sin(0.517*obs[8]))
    - obs[15]
    + obs[2]**2,
    obs[15]*obs[2]*obs[6]**2 - obs[15]*jnp.sin(jnp.sin(0.517*obs[8])) - obs[15] - jnp.sin(0.517*obs[8]) + jnp.sin(obs[2]**2*obs[8]),
    obs[2]**2*obs[8] - 0.517*obs[8] - jnp.sin(jnp.sin(0.517*obs[8]))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[563.8375  130.74269 739.89764 649.41846 615.92816 171.52023 565.0603
 104.18243 467.89767 581.8578 ]


### CGPAX

In [ ]:
brax_env = BraxGymnaxWrapper('walker2d', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([(obs[6] - obs[10]),
jnp.cos(obs[6] - (obs[16]*obs[2])),
(obs[1] - obs[16]),
jnp.cos(obs[6] - (obs[16]*obs[2])),
((obs[13]*obs[9] - obs[3]) * (obs[13]*obs[9] - obs[3])),
jnp.cos(obs[6] - (obs[16]*obs[2]))]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1010.26733 1008.0946   916.777    995.9993   997.39526  985.2489
  872.1092   972.6999   882.25385 1005.94135]


In [ ]:
brax_env = BraxGymnaxWrapper('walker2d', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([jnp.cos(obs[9]),
(jnp.sin(obs[16]) - obs[1]),
jnp.cos(obs[0]),
(jnp.cos(obs[0]) - obs[8]),
(((obs[12] - obs[12])*obs[7]) - jnp.sin(obs[6])),
jnp.cos(obs[4])]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[  57.81344   56.8279  1138.7505  1149.473   1144.2706  1146.2981
 1125.3494  1136.2214  1136.0172  1146.6349 ]


In [ ]:
brax_env = BraxGymnaxWrapper('walker2d', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([(1 + jnp.cos(obs[0])),
jnp.cos(obs[2]),
((obs[12] - obs[7])*(obs[12] - obs[7])),
jnp.cos(obs[0]),
((obs[16] - obs[1]) + (0.1 - obs[10])),
obs[14]]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1442.1514  1527.7806  1470.4994  1477.1031   158.36446 1518.1287
 1464.3116  1515.7708  1495.168   1487.8435 ]


## Half cheetah

Action space: $[-1, 1]^6$, where the first action is the torque applied on the back thigh rotor, the second action the torque applied on the back shin rotor, the third action the torque applied to the back foot rotor, the fourth action is the torque applied on the front thigh rotor, the fifth action the torque applied on the front shin rotor, and the sixth action the torque applied to the front foot rotor.

Observation space: $ℝ^{18}$

0. z-coordinate_mass
1. w-orientation_front_tip
2. y-orientation_front_tip
3. angle_back_thigh_rotor
4. angle_back_shin_rotor
5. angle_back_foot_rotor
6. velocity_tip_along_y-axis
7. angular_velocity_front_tip
8. angular_velocity_second_rotor
9. x-coordinate_front_tip
10. y-coordinate_front_tip
11. angle_front_tip
12. angle_second_rotor
13. angle_second_rotor
14. velocity_tip_along_x-axis
15. velocity_tip_along_y-axis
16. angular_velocity_front_tip
17. angular_velocity_second_rotor

The reward consists of two parts:

  - *reward_run*: A reward of moving forward which is measured as (x-coordinate before action - x-coordinate after action) / dt.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as *-coefficient x sum(action<sup>2</sup>)*


   The episode terminates when episode duration reaches a 1000 timesteps.

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("halfcheetah", 'generalized')
# env = EpisodeWrapper(envs.get_environment(env_name="halfcheetah"), episode_length=1000, action_repeat=1)
T = 1000

In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )
        # env_state = env.step(
        #     env_state,
        #     action
        # )
        # reward = env_state.reward

        return (obs, env_state, done), (obs, reward, action, done) #, env_state.pipeline_state.x.pos, env_state.pipeline_state.q)

    return _step

def visualize_trajectory(all_obs, treward, actions, all_dones, all_pos, all_angles):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  print(reward)
  all_obs = jnp.array(all_obs)
  all_pos = jnp.array(all_pos)
  all_angles = jnp.array(all_angles)
  plot_list = []

  for i in range(0, T_term, 50):

    print(
            f"Timestep {i} : "
            #f"Height: {all_obs[i, 0]} | "
            f"x-coordinate: {all_angles[i][0]} | "
            f"z-coordinate: {all_angles[i][1]} | "
            #f"Thigh angle (rad): {all_angles[i, 2]} | "
        )
    xz = all_pos[i][:, [0, 2]]
    plot_list.append([i, xz])

  print(
          f"Timestep {T_term} : "
          #f"Height: {all_obs[T_term, 0]} | "
          f"x-coordinate: {all_angles[i][0]} | "
          f"z-coordinate: {all_angles[i][1]} | "
          #f"Thigh angle (rad): {all_angles[T_term, 2]} | "
        )
  xz = all_pos[T_term][:, [0, 2]]
  plot_list.append([T_term, xz])
  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  # for i in range(0, 1000, 10):
  #   xy = all_pos[i][:, :2]

  #   plt.plot(xy[:, 0], xy[:, 1])
  #   plt.scatter(xy[:, 0], xy[:, 1])
  #   plt.title("State at t="+ str(i))
  #   plt.gca().set_aspect("equal")
  #   plt.show()

  # xy = all_pos[:, -1, :]
  # x = xy[:, 0]
  # y = xy[:, 1]
  # plt.plot(jnp.arange(1000), x, label = "x")
  # plt.plot(jnp.arange(1000), y, label = "y")
  # plt.legend()
  # plt.show()
  # plt.plot(x, y, label = "tip trajectory")
  # plt.legend()
  # plt.show()

  plt.plot(all_rewards, label="reward")
  plt.plot(all_actions, label="actions")
  plt.legend()
  plt.show()

def compute_and_visualize(key, obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, dones, all_pos, all_angles) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, dones, all_pos, all_angles)

### Kozax

In [ ]:
brax_env = BraxGymnaxWrapper('halfcheetah', 'generalized')
policy = lambda obs: jnp.array([
    obs[4],
    obs[0] + obs[3],
    obs[1] - jnp.cos(jnp.sin(jnp.cos(obs[2]))),
    -1.69*obs[1]*obs[15]*obs[7],
    obs[2]*obs[7]*obs[8],
    obs[3]*jnp.sin(jnp.sin(obs[3]))
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[3513.517  3529.7107 3476.651  3539.7864 3567.2534 3509.356  3504.1128
 3530.056  3506.142  3537.2358]


In [ ]:
brax_env = BraxGymnaxWrapper('halfcheetah', 'generalized')
policy = lambda obs: jnp.array([
    3*obs[0],
    2*obs[2]*(obs[0] + obs[2]),
    -0.985,
    obs[0]**2*obs[1],
    obs[4]*(obs[2] + obs[4]) - obs[4],
    (obs[11] + obs[3])*(obs[2] + obs[7])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[3619.2832 3620.1467 3631.4543 3601.2593 3627.2134 3619.0261 3597.0898
 3618.9932 3610.2515 3627.8965]


In [ ]:
brax_env = BraxGymnaxWrapper('halfcheetah', 'generalized')
policy = lambda obs: jnp.array([
    -0.609,
    obs[4] - jnp.sin(obs[0]) - 1.2,
    4*obs[0]*obs[7],
    obs[2]*obs[4]*obs[6]*obs[7],
    obs[2]**3*obs[4],
    -1.17
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[4565.6045 4809.5576 4920.3096 4987.2056 4755.5522 4939.1978 4991.4927
 4967.416  4953.4077 4817.546 ]


### Modi

In [ ]:
brax_env = BraxGymnaxWrapper('halfcheetah', 'generalized')
policy = lambda obs: jnp.array([
    obs[7] + (obs[5] + obs[7]*obs[9] + obs[7])*jnp.sin(jnp.sin(obs[0] + obs[1] + obs[5]**2 + obs[9])) + jnp.sin(obs[7]),
    0,
    obs[5] + obs[7]*obs[9] + obs[7],
    0,
    0,
    obs[7] + jnp.sin(obs[7])
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[4704.554  4739.2046 4675.595  4786.4453 4837.469  4718.989  4712.607
 4756.5454 4681.5615 4794.583 ]


In [ ]:
brax_env = BraxGymnaxWrapper('halfcheetah', 'generalized')
policy = lambda obs: jnp.array([
    -1.8*obs[4]*obs[5] + jnp.sin(obs[2]),
    obs[0] - 3.59*obs[4]*obs[5]*(obs[5]*(obs[0] + obs[9]) - obs[7]*(obs[11] + obs[12]) - 0.602) - 0.898*jnp.sin(obs[2]) - 0.898,
    obs[5]*(obs[0] + obs[9]) - 0.602,
    0,
    obs[0]*(obs[0] - 1.8*obs[4]*obs[5]*(obs[5]*(obs[0] + obs[9]) - obs[7]*(obs[11] + obs[12]) - 0.602) - 0.898*jnp.sin(obs[2]) - 0.898),
    obs[0] - 1.8*obs[4]*obs[5]*(obs[5]*(obs[0] + obs[9]) - obs[7]*(obs[11] + obs[12]) - 0.602) - 0.898
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[4472.0464 4506.6323 4509.0625 4537.7793 4554.5303 4492.277  4454.577
 4480.199  4458.263  4462.0723]


In [ ]:
brax_env = BraxGymnaxWrapper('halfcheetah', 'generalized')
policy = lambda obs: jnp.array([
    obs[7] + jnp.cos(obs[14]*obs[7] + jnp.cos(obs[7]*obs[8])) - 1.23,
    0,
    obs[7] - jnp.cos(obs[14]*obs[7] + jnp.cos(obs[7]*obs[8])),
    0,
    -0.704*obs[7] + 0.296*obs[9]*(obs[7] - jnp.cos(obs[14]*obs[7] + jnp.cos(obs[7]*obs[8]))) - 0.704*jnp.cos(jnp.sin(obs[7] - 1.23)) + 0.865,
    0
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[3958.9424 3970.381  3970.2935 3979.6753 3992.2388 3967.8606 3960.3457
 3967.148  3954.051  3971.694 ]


### CGPAX

In [ ]:
brax_env = BraxGymnaxWrapper('halfcheetah', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([(obs[9] + jnp.cos(jnp.sin(0.1))),
jnp.cos(0.1),
((obs[13] - obs[4]) - obs[11]),
obs[5],
(((obs[13] - obs[4]) - obs[11]) - jnp.sin(obs[8])),
obs[0]]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[4752.1465 4724.203  4832.36   4293.845  4353.0664 4857.8555 4888.904
 4665.2275 4838.656  4495.7603]


In [ ]:
brax_env = BraxGymnaxWrapper('halfcheetah', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([jnp.sin(jnp.sin(jnp.cos(obs[17]))),
(jnp.cos(jnp.sin(jnp.cos(obs[17]))) + jnp.cos(obs[17])),
((obs[0] - obs[8]) - obs[15]),
jnp.sin(jnp.sin(obs[5])),
(obs[13] + (obs[15] - obs[16])),
obs[0]]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[2151.236   2648.3196  1667.2136  1283.3256   179.19232  793.21216
 1855.6768  2489.5513  1457.6787  1948.9033 ]


In [ ]:
brax_env = BraxGymnaxWrapper('halfcheetah', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([jnp.cos(obs[9]),
jnp.cos(obs[4]),
(obs[2] - (jnp.sin(obs[16]) - ((obs[15] + obs[15]) + (obs[15]*obs[13])))),
obs[5],
obs[14],
obs[5]]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[ 9241.713   9674.164   9599.209   9851.3955 10129.716   9814.777
 10112.454  10244.052   9801.973   9345.091 ]


## Ant

Action space: $[-1, 1]^8$, where the first action is the torque applied on the rotor between the torso and front left hip, the second action the torque applied on the rotor between the front left two links, the third action is the torque applied on the rotor between the torso and front right hip, the fourth action the torque applied on the rotor between the front right two links, the fifth action is the torque applied on the rotor between the torso and back left hip, the sixth action the torque applied on the rotor between the back left two links, the seventh action is the torque applied on the rotor between the torso and back right hip, the eighth action the torque applied on the rotor between the back right two links.

Observation space: $ℝ^{27}$

0. z-coordinate of torso
1. w-orientation_front_tip
2. x-orientation_front_tip
3. y-orientation_front_tip
4. z-orientation_front_tip
5. angle_torso_first_link_front_left
6. angle_two_links_front_left
7. angle_torso_first_link_front_right
8. angle_two_links_front_right
9. angle_torso_first_link_back_left
10. angle_two_links_back_left
11. angle_torso_first_link_back_right
12. angle_two_links_back_right
13. x-coordinate_velocity_torso
14. y-coordinate_velocity_torso
15. z-coordinate_velocity_torso
16. x-coordinate_angular_velocity_torso
17. y-coordinate_angular_velocity_torso
18. z-coordinate_angular_velocity_torso
19. angular_velocity_torso_front_left_link
20. angular_velocity_front_left_links
21. angular_velocity_torso_front_right_link
22. angular_velocity_front_right_links
23. angular_velocity_torso_back_left_link
24. angular_velocity_back_left_links
25. angular_velocity_torso_back_right_link
26. angular_velocity_back_right_links

The reward consists of four parts:

  - *reward_survive*: Every timestep that the ant is alive, it gets a reward of 1.

  - *reward_forward*: A reward of moving forward which is measured as (x-coordinate before action - x-coordinate after action) / dt.
  
  - *reward_ctrl*: A negative reward for penalising too large actions. Measured as *-coefficient x sum(action<sup>2</sup>)*

  - *contact_cost*: A negative reward for penalising too large external contact force. Calculated as calculated *0.5 * 0.001 * sum(clip(external contact force to [-1,1])<sup>2</sup>)*  


   The episode terminates when episode duration reaches a 1000 timesteps or when the y-orientation (index 3) in the state is **not** in the range `[0.2, 1.0]`.

### Visualize best solution

In [ ]:
env = BraxGymnaxWrapper("ant", 'generalized')
T = 1000

In [ ]:
def step_fn(policy):

    def _step(carry, t):
        obs, env_state, done = carry
        action = policy(obs)
        obs, env_state, reward, done_new, _ = env.step(
            env_state,
            action,
            None
        )

        done = done | done_new.astype(bool)
        reward = jnp.where(done, 0.0, reward)

        return (obs, env_state, done), (obs, reward, action, done) #, env_state.pipeline_state.x.pos, env_state.pipeline_state.q)

    return _step

def visualize_trajectory(all_obs, treward, actions, all_dones, all_pos, all_angles):
  # Convert to arrays
  all_actions = jnp.array(actions)
  all_rewards = jnp.array(treward)
  reward = jnp.sum(all_rewards)
  T_term = jnp.where(all_dones.any(), jnp.argmax(all_dones), 1000)
  print(reward)
  all_obs = jnp.array(all_obs)
  all_pos = jnp.array(all_pos)
  all_angles = jnp.array(all_angles)
  plot_list = []

  for i in range(0, T_term, 50):

    print(
            f"Timestep {i} : "
            #f"Height: {all_obs[i, 0]} | "
            f"x-coordinate: {all_pos[i][1, 0]} | "
            f"x-coordinate: {all_angles[i][0]} | "
            f"z-coordinate: {all_angles[i][2]} | "
            #f"Thigh angle (rad): {all_angles[i, 2]} | "
        )
    xz = all_pos[i][:, [0, 2]]
    plot_list.append([i, xz])

  print(
          f"Timestep {T_term} : "
          #f"Height: {all_obs[i, 0]} | "
          f"x-coordinate: {all_pos[T_term][1, 0]} | "
          f"x-coordinate: {all_angles[T_term][0]} | "
          f"z-coordinate: {all_angles[T_term][2]} | "
          #f"Thigh angle (rad): {all_angles[i, 2]} | "
        )
  xz = all_pos[T_term][:, [0, 2]]
  plot_list.append([T_term, xz])
  if T_term < 1000:
    print("Terminated at timestep:", T_term)
  else:
    print("No termination (ran full 1000 steps)")

  plt.plot(all_rewards[:T_term], label="reward")
  plt.plot(all_actions[:T_term], label="actions")
  plt.legend()
  plt.show()

def compute_and_visualize(key, obs, env_state, policy, T):

  # JIT the whole loop
  (_, _, _), (all_obs, treward, actions, dones, all_pos, all_angles) = jax.lax.scan(step_fn(policy), (obs, env_state, False), jnp.arange(T))

  visualize_trajectory(all_obs, treward, actions, dones, all_pos, all_angles)

### Kozax

In [ ]:
brax_env = BraxGymnaxWrapper('ant', 'generalized')
policy = lambda obs: jnp.array([
    obs[4]*(obs[2]*obs[4] - obs[7]),
    obs[11]*(2*obs[2] - obs[3]),
    -0.151,
    jnp.cos(jnp.cos(jnp.cos(jnp.cos(obs[1])))),
    obs[3],
    obs[0]**2*jnp.cos(obs[0]),
    obs[0]**3,
    obs[2]**2*obs[3]**2
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1268.3302  1322.4113  1298.6351   755.531    879.9645  1301.4751
  966.07404  131.09488 1297.0237   187.68376]


In [ ]:
brax_env = BraxGymnaxWrapper('ant', 'generalized')
policy = lambda obs: jnp.array([
    jnp.sin(obs[3])*jnp.sin(obs[18]*obs[3]),
    obs[4]**2,
    obs[2]*jnp.sin(obs[12]*obs[5]),
    obs[2]**3*obs[7],
    obs[2]**3*obs[20],
    obs[3]**4,
    obs[6]*jnp.sin(obs[2])**2,
    obs[2]*obs[24]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1002.08905    39.966995   45.969368  995.64465   332.71567   321.06247
  984.01654   100.19222   979.88916    49.22131 ]


In [ ]:
brax_env = BraxGymnaxWrapper('ant', 'generalized')
policy = lambda obs: jnp.array([
    obs[26]*obs[3]**3,
    0.00569,
    2*obs[2]**2,
    obs[2]*obs[4]**3,
    -0.155*obs[2]**2,
    obs[13]*obs[21]*obs[3]**2,
    obs[2]*obs[3] - obs[3]**2,
    obs[2]**3*obs[9]
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1072.9198  1032.5925  1028.2296  1031.4852  1031.7964  1019.458
 1005.3097  1055.7407  1020.6965  1018.43353]


### Modi

In [ ]:
brax_env = BraxGymnaxWrapper('ant', 'generalized')
policy = lambda obs: jnp.array([
    0,
    obs[0]*(obs[12]*obs[2]*(-obs[0] + jnp.cos(obs[12] + obs[13]*obs[18]**2*(obs[12] + obs[14]*obs[2]) - obs[8])) - jnp.cos(-obs[12] + obs[17] + obs[18]))
    + obs[12]*obs[2]*(-obs[0] + jnp.cos(obs[12] + obs[13]*obs[18]**2*(obs[12] + obs[14]*obs[2]) - obs[8]))
    + jnp.cos(-obs[12] + obs[17] + obs[18])
    + jnp.cos(obs[12] + obs[13]*obs[18]**2*(obs[12] + obs[14]*obs[2]) - obs[8]),
    0, 0, 0, 0, 0, 0
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1054.484  1149.2557 1094.1892 1163.0001 1158.7762 1168.7806 1077.991
 1163.8214 1117.8789 1093.5663]


In [ ]:
brax_env = BraxGymnaxWrapper('ant', 'generalized')
policy = lambda obs: jnp.array([
    0,
    obs[1]**2*jnp.sin(0.0673*jnp.cos(obs[12] - obs[26]*(obs[8] + jnp.sin(obs[9]*(obs[0] + 0.0673)))))
    + obs[1]*obs[11]*obs[8]*jnp.sin(obs[1]**2*jnp.sin(0.0673*jnp.cos(obs[12] - obs[26]*(obs[8] + jnp.sin(obs[9]*(obs[0] + 0.0673))))))*jnp.cos(obs[26])
    + obs[1]*jnp.sin(obs[1]**2*jnp.sin(0.0673*jnp.cos(obs[12] - obs[26]*(obs[8] + jnp.sin(obs[9]*(obs[0] + 0.0673))))))*jnp.cos(obs[26])
    + obs[1]*jnp.sin(obs[1]**2*jnp.sin(0.0673*jnp.cos(obs[12] - obs[26]*(obs[8] + jnp.sin(obs[9]*(obs[0] + 0.0673))))))
    + obs[1]*jnp.sin(0.0673*jnp.cos(obs[12] - obs[26]*(obs[8] + jnp.sin(obs[9]*(obs[0] + 0.0673))))),
    0, 0,
    obs[1]*obs[8]*jnp.sin(obs[1]**2*jnp.sin(0.0673*jnp.cos(obs[12] - obs[26]*(obs[8] + jnp.sin(obs[9]*(obs[0] + 0.0673))))))*jnp.cos(obs[26]),
    0, 0, 0
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1192.3944 1196.6754 1192.389  1120.0396 1117.3763 1176.2704 1192.2329
 1196.8429 1184.564  1182.9313]


In [ ]:
brax_env = BraxGymnaxWrapper('ant', 'generalized')
policy = lambda obs: jnp.array([
    0,
    0,
    0,
    jnp.cos(obs[6] - (-obs[10] - obs[13] - obs[19] + obs[6])*(-0.0277*obs[1]*obs[19] - 0.0277*obs[14] - 0.0554*obs[20] + 0.0277*obs[26])),
    0, 0, 0, 0
])
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[1184.4277  1166.5747  1264.4058   932.39343 1234.8143  1295.899
 1315.7355  1196.2839  1244.1534  1080.7595 ]


### CGPAX

In [ ]:
brax_env = BraxGymnaxWrapper('ant', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([(obs[16]*(jnp.sin(obs[6]*obs[15]) - obs[4])),
jnp.sin(obs[10]),
(obs[24]*obs[4]),
jnp.sin(obs[10]),
(((jnp.sin(obs[6]*obs[15]) - obs[4]) - (jnp.sin(obs[6]*obs[15]) - obs[8]))*jnp.sin(obs[15])),
obs[6],
(obs[6]*obs[15]),
obs[6]]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[905.2866   492.24573  698.25146  222.80656  822.3367   888.8482
 812.38336  852.8141   820.6327    18.145443]


In [ ]:
brax_env = BraxGymnaxWrapper('ant', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([(jnp.sin(obs[12]) - jnp.sin(obs[2])),
obs[3],
(jnp.cos(obs[2])*jnp.sin(obs[3]*obs[5])),
jnp.cos(obs[0]),
jnp.sin(obs[2]),
0.1,
jnp.sin(obs[12]),
obs[11]]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[ 351.88477  333.77856  665.8652  1496.457   1390.1401  1322.6346
 1496.0708  1385.0074  1222.7651  1382.7599 ]


In [ ]:
brax_env = BraxGymnaxWrapper('ant', 'generalized')
policy = lambda obs: jnp.tanh(jnp.array([obs[8],
jnp.cos(obs[4]),
obs[2],
0.1,
jnp.cos(obs[4]),
jnp.cos(obs[4]),
obs[7],
(0.1 + obs[2])]))
key = jr.PRNGKey(0)
reward_list_inv_pen = repeated_evaluation(key, policy, brax_env)
print(reward_list_inv_pen)

[  -5.2547126 1078.1351    1022.7895     984.5019     969.36835
 1024.9307    1083.9714      -2.6536703 1034.8174    1043.1321   ]
